# Kaggriculture Cow Placement | Historical LB 2531

A cow in the shed is not yet a productive cow. It must be picked up, taken to a
suitable tile and placed before the farm can earn milk revenue. This notebook
packages a complete historical agent and examines a small repair to that chain.

**Run All creates `main.py` and `submission.tar.gz`.** The default source matches our
S59 submission byte for byte. No GPU or external dataset is required, and running
the notebook does not submit anything to the leaderboard.

**2531.0 is historical:** our submission `55964457`, uploaded on 2 September 2026,
showed that public LB score when checked on 8 September. It is neither a new score
for this notebook nor a forecast for a fork. The archive is a reusable baseline,
not a claim that this is today's strongest agent.

The complete farming and adaptive-market controller comes from
[Boatlee's V29-R1, version 1](https://www.kaggle.com/code/boatlee/v29-r1-adaptive-market-hysteresis?scriptVersionId=346682136),
released under Apache 2.0. Boatlee attributes its production schedule to public
RngRng replays. Our S59 changes are a carried-animal placement repair and three
late-market settings. The market controller itself is not our invention.

## Quick start

1. Enable Internet for setup and run all cells. The agent itself is offline and uses only Python's standard library.
2. Read the guard tests and complete-game results. They check mechanics and packaging, not a 2500-point performance guarantee.
3. Save a completed notebook version. Select its `submission.tar.gz` on the competition's Submit Agent page, or download and upload that file yourself.

The full source is below, followed by tests and packaging. You can edit it directly;
the output will then be labeled modified rather than historical.


In [1]:
import importlib.metadata
import subprocess
import sys
ENGINE_PIN = "1.32.7"
try:
    installed = importlib.metadata.version("kaggle-environments")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != ENGINE_PIN:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
                             "kaggle-environments==" + ENGINE_PIN], capture_output=True, text=True, timeout=240)
    if result.returncode:
        print((result.stdout + result.stderr)[-2000:])
        raise RuntimeError("Engine setup failed. Enable Internet and retry setup.")
assert importlib.metadata.version("kaggle-environments") == ENGINE_PIN
SMOKE_SEEDS = (59001, 59002)
print("Test engine:", ENGINE_PIN, "| CPU only")


## 1. Repair the missing step, not the whole route

The frozen route can request `PLACE COW` on bare soil. That action cannot create
a pasture. If the actor already carries a cow and its next scheduled action is
`PASS`, S59 uses two calls:

| Call | Scheduled action | S59 action | What must be observed |
|---|---|---|---|
| Now | PLACE COW | BUILD_PASTURE | Carried cow, bare tile, next trace slot is PASS |
| Next | PASS | PLACE COW | Empty pasture exists and the actor still carries the cow |
| Later | Original route | Original route | No further placement-repair delay |

Sheep use a pasture too; geese require a coop. An animal merely requested from
the shop does not qualify. A busy next trace slot does not qualify either.
These checks keep the change local to an already-started transaction.

There is an important gap: **a free trace slot is not necessarily an available
physical turn.** The repair has no day-boundary guard. A build at the final hour
can leave the animal unplaced when the day ends. A failed build also ends the
attempt; this is not a general route planner or a guarantee of placement.

The tests below call the actual function from the packaged agent, with synthetic
observations. They cover successful completion, refusal conditions and the
day-boundary counterexample. Synthetic observations are not downloaded replays.


## 2. Keep extra sales inside the available stock

The inherited market controller first protects pickups and already scheduled
sales. Extra selling is bounded by the remaining shed stock, an item reserve,
a tranche size and a per-item budget. Prices and public market conditions decide
when an otherwise affordable extra sale is worth attempting.

For example, 20 units of milk minus 3 for pickup, 5 already offered for sale and
a reserve of 2 leave at most 10 extra units. A tranche of 4 restricts the extra
order to 4. Milk still being carried or expected from a future harvest does not
increase the observed shed stock.

S59 changes only these inherited settings:

| Setting | Boatlee V29-R1 | S59 | Meaning |
|---|---:|---:|---|
| `reserve_tail` | 3 | 0 | Reserve used at steps 648–683, before other adjustments |
| `tail_tranche` | 5 | 8 | Extra tranche allowance from step 684 |
| `max_extra_per_item` | 18 | 24 | Extra-sale budget per item |

This is not a claim that aggressive selling always wins. The near-mirror guard
can disable extras, prices can block them, and the original route can still
contain unsuccessful orders. A full ten-order queue can also consume internal
extra-sale budget without adding an order. We preserve these historical behaviors
and expose them in tests instead of silently improving the scored source.


## 3. Complete agent source

This cell contains the entire policy, including the inherited compressed action
schedule. That schedule is data decoded with JSON, not executable code fetched
from elsewhere. The controller remains readable below it. Expand the cell to
edit `_animal_place_repair_action` or `_AM_CONFIG`.

The following materialization cell restores the historical mixed line endings
only when the normalized source is unchanged. Real edits are kept.


In [2]:
SOURCE = r'''"""ScoreBand-top20-Rank4 for Kaggriculture."""
import base64
import copy
import json
import zlib


_ACTIONS = json.loads(zlib.decompress(base64.b85decode(b'c-q}v(QX_`a{L!Q^FcElQk37g(wtW~?NXqm8=M!yVga9Fz&Jmw{bu;zEiK8Op3caK$gCdi`kq8k!|Cd(tg6h&$jG1m=i=Xf`StIA`}N|Ve!6&n_u<3E=gW(K|K&gb^}oLS;>(YJ|K->J_}hPf`T3`d?>_zY=l2iyZ$7>LcyW1g_<n!)<=^Y8&!7MIa{eVh-amZ&KL3^v`*-h-f4;l@eEImzcR#-U;qL4IyuLqNTyDd!KYYL6zkm9V@83S`FWeu#e#?jb{>|~BFBiZ5VgK&_$BWC&cKIrY{fCe9H~BI{Z(slU)BER9ZoYoe(}#{PKKs@3<MXMI&pLkJ^fgbDusMC)@u%-Ty?yuQ=P%3n@zdA0Cy#so?(WBr)4QKMckw0j=RZy0;`wlRUmoy!kKcT01e@pe2-dX;D{}X7|FBq+(+A-dfp75oZaGPDWO9`rzvL25+hUGqBYXUzHz~nP$yy#Q@oARkOY}UM(R6zQ+ii{IIzDQexoMDb3P*+$7H@gHp%wbm*CXL{4}+Oo{N35cm#@&tP#-_FmNmsukP~jPh~IFfW}Exz{~o_>xp>(^6(%2SWVA4G8T_e5Ul@KWeRXZd;-?p%Im6V2pGK!{fA?Xz{=fZMYu=7?GktA7ZL{ChpIDu(&PzT122R%We<R;}{2`f<wZ4Bj4Xcei`GI}y!l}4QonGmuF24EcXom~9UoM+I9G(v8)xPfj-Mjtkk3auu|M2ncySM)mJhH`C&A+|mq5gL7&Ytr13PH~SuGi`N!E0&p#mU{I<D1~taYs)tHS$%#9Sjy~dF6)7Jz4^Cmh;IgTEfyr0UNm#5;4o_%rrSv^zg{80>hK1iM^{DC)LwgkG`#O4Scx&wA=wh$0*q7+y1x1{k={1_swVD>PnB#eA{!bGcn0^t!AZV-mJXLP9QPz3V-BpTihBuisPLEV%HXy*6>mMYtV5~D)ErL%T~Hr3)YRoX%fLx`v%SkxP&Sb7nc<TT9wF+tn766F#M^#u&vQrKF5%EN5?m?P0;Z4;xuOS`P;w0d-#hQFLM=+(Rzq~HRfPV>@>nrLw$Y?6yC0R7PU;4T+`E9O>^xm8am6a)JgBGAw1Kra0Vz|<;&E|YL2{jh%W55`F&u3PsU{|2z&DsDf!(zkSj9>SWM~316xMpy@NY`iFF~2gygj+6;B=F9)L&J%-uq>N8OR&J|P&rbl&q}&u<gxB1Xr$9Q++G8VLNxL`UOxNbbjYk~?<=L&MVHvRp)YN`ZxdsUs6R>~6vN)TesGEnt3#ew(5*79KnL)%o15<BY@yG0%F)8L`*yxMC>SX}$eMJqg)*7eKL`;+a|6-U4%yX5Sb2HfKOo-#pyEPqHDaKyHn%HB)VqTR7&bU7s;?HR6SV$)(AWb9hIo6IL_sDHW1?TlL);XZyHyOn<B4p9(ry$p;h5049HVj+Pw;5;4HFLD*c&F9F%GnzKdD8sK`A@Dx8Q5gZTrr=$MJabhNhQM%rMPjuN_qUcM6%d`n%10N19xl56<7i`X6LB2#CvDp+-W~oosd1ta-zy&cpa0~Gx-?#c=wO)e&fZ+QE=AS%ha-qf&g4Zi21b0K6!xh1>x`)-qxjeE<Ik^|}i);IKz(q=-u$l`aw%_uK5oG?O1y%A$$u&6$zQu*&uvp=%iwiV8;_*`hz6My1n8-i}!0gKZ@bTg9@ZJ95;jh3R;yK2Bm@W@9J_n<T09jmzQ86vp^A$gv7V_%`Y;c**zLT>)vdD8Jw+SjFD~sZ!!pc$-VDTM>W)9#>AnWP85~$Tc^gV1b^JnTO<wZ0Qx=L)f@j`N&w~QffbB06~%ozkoDbzN+*yuFbB@fz@Ll}BtBfi!;#P<<rZfUSZh|EXN5GYCkFR&!h@odte;Ruf#os98Hj4}S;vv|Rt05x0Tk@7R-k?@^k^;&!=aAf?a+D;1i@>!8x^=}M+<8Xif#b7os<ngh|-aWdbfEPhM{b>kAI{!oj{&K0jxe&$2$G@iV2XphLn;`x$nPm2vPSpZu!x9Qj91VDh$)ML9@^WPaXa^2Nv@MTUa7-Z`-$(5Z@_KF8Gdc%xHc?cFX4ry1X)Ls?oq<*y3vA+V;!C7lxzcI#nBC%0=rR)qYq9b1L^vO7Rt?2CeX`WVzl|F*e}}w)8h!a-I0v_>kv5XfHz}p3#hx~xx$S^z(XJUkOWSpPCJK5YKV})DOXg*PTQgtaEAt$V?-6}0u#l{xy+qUo(OpCl*vWc``7(LR+pu0BK~<3#9sf4g*1coF<O1slV-+AohkrlY9SI1CJxc%TMtMlgI<jGj3`%G$C9GyM!g-V#&gszuOU4{B;=FH}?>-(%7qk3U(Jf0lyvo&w1efd}<LOO0LV*s{|73(1eVk27c8S--I=Q0h_@Dl7`XS9ItdXTSJ_TO|(+7_~sIX%_aA59bV&^gYL;Pza%r`Nn@hDq`aV+wqyiDK!uhTWN`2zzw;ql<V8W?;^buFZFm}prX{lrt0aZJueUW1$x<Hoot0*nAKfrDEOV8l;Ot|vti#W=aIS@W@x{0iA-JhRdXmMGc@1c30nTNJtBHT33>kPiKtK!48yZx?7lEVeAA!!kx@Yevv@t$b*+0I?O=W*Qm!Xut+wctvmD5lyF79y4ao$cuwujKF;&>)lbrv*uwX`7CAtk!sgbf2qCcai^qS+F3=oz(6AQGEfWBO|W*7mDeaJ+6^T+ww=MU#HLX^kby5g4Dx*D|M2$RpTFwjz+i_`<*@H@`lX7%|1_PCpWkc=!mh&XR6)3)oI<pGFCjF=oDxg!VnBurWDFL{yjz(srWvRPH~{PgJYswx?zGuyiKPeRjk;X!>?i8{ilq?qKd!z60J*d7r&+EB*?`fGr#FHp8?6YBHO@WzZG;xI%%>EWt!~|d@kSv+Gp<~v{tI9s?BWw))4rTByegnjClSbU-|Z_&#Kot3abC3Lco6}wEkkE%!e<>@g*zuE>Cku(lbXz8SH=>dTi#ez7Pph60;`9f3^y}lU=>0oa9~QpnRmFL33l|IJsA$7xx#KpcrOl3psWnb5|IoHgl3AiNVS~m$_QG#nN&#=%4+b{H!YpMT`i$cYU?<Uy<EFCE*PR^vo9T_;oFE{uaf13;XODwP{3~<O;s(CEWtGhWH83&vG51IlEsQ?RTXxfD3))Lq|@x)F9fcq)aG4@#wp3xlIZIWdcmOB`7_S)C^XV$3j;2fF=;PUUufSLC_rDXs-1@F&ol_Z2n52WSGY{3Rz;Q7AgKvQx~8yp<TlX+MFG`m=TKRm)U$dNY|6k=kY$TnA-?m3)&TRoe0770$wRz`V#x3$XQ`#L**uf4D@;7;PUJ}duW`nkdy&RB`4%Y0Use$p!X(lV>(VR0?aV6y1wm*U>%v&%fbq<R&`f5-Ku<@+H#8d1ijHfDJ+b7fp*q)1yGOT_J;Kr3yW|Rs67;E!D6Qbl#98VyW|sHrjvjuoap{K~HIi=;8i72`+HBO$JUTykD%Bdx7^rsc{cxtz>(vLd8lhla;yJ`73Tn?PD$l`&+m(g1DG$N=I}=s7O(U=Cl^Ornl}s*Q(jE_v4VDr>xc|2NQ>Ur3g+Fn%Y~UktcXaLwG0i`DMbbK!+yTOWhJR}o5O=1UBp9`1kZ>&FFDsE0864oRPnHM%&#8;A=p$ZNXs%#90J#@F=QFYCv3d9nfDJIQ>G|^+`)c|=KJw#FQsIaX8lZUm8`*0S!j^Gfqv7!(N35^&xd`_due#l}3uO-o;0U84b?)<zQ9jag3%1v8s3;_=urR7O>%9W*mlU`XnA<Xz#Z)R0RvUN8qD$%P`(+Ffi=+i5?RbRH1IAs<LiIwzVo00h;3%E0>+<tuY>Bobi3S*N8@HQr_^-v6Tep^}d1Be>-Lm;DxYWEH@#3q>LSmjmbZZwexh3`(KV2)%jRSFn<KPk^^y69Um|5Deo71KXJPlwRahou^nXePs1HMhO{Ti*D{SZwtx9la}a$_8bFldq086}9wwrU@c85ma&p7EWBaucx==LOLcL<vDMu<J9jQBce6Ku2+roy`h3)=>|w(anPsZV+u#a3UioVpv<uNLebWT*?8Z<W%Q<y4nMIn}o9Tj76cTF@6g&WWIcwy6$-qts<`zhVPJUmXY%3EoT_Zn3p@UUA9iT-`+cF_5o6HtQ&LQmpDZ?mDL&9fQ3ak?2Dxo#Y}eLjcsKs2|RIOuf3%V&}tqdPd?IdI5kANE7w?+Ax;ouP3T}<eZZhktg8?h<5gbc;HV_5RQR(RiO3a<1AL+ANzgPv84@GP2eib5k)3RJ#N>8iykyMN0EvpYvFc_w)-W#UgMuK|n|PM@@I^%wHg&SBCD~7(kmR0dh@Z4Z`(CDtNGI26@tz`H4bjBlkpYyFQi({B3l)>^O`_*cm4Kd3KXg=MC&=b@wNS0>#u6MX7^ej=4w0Ff+9sn<{|6@r>HZF?50{IbY%R_QfqG%L1;!34_0wYNe*t%84H&!`>T#Gsy@}-I`e;VR!=`mBg*qqL#RKH(P;B((?LQ0dlB%w8r2JUdrD%=7nPdLJ^6F`}_XXC-?lef90;bSzPofOazR4;r2}GRHxM!*LaV(*^zu~l^3QJPltT|<x;KphtNHUCs4@iR~#0_UNIF*H>l`}RVq3XXMF~L{O{yK5@<c4Q~ldnw71h`W90T|P{XrqXJ#Bu-tt0C6NZSVOS8I^F1qf{kx1JZhi!ImbJu{LO-LoQfRd%nVb*=T??f3n<bh(Q{*?<o1F!FgQ}gT*>ze@vAnM4h9T^!K@Jy{KCAC^mz#o@)zMoVie?5X7`PHIIa!tShx@<5#U>sTHW~42r^1$#GMlE1>g|NL?|;x+}0g4{I{ahU8K~`U2FJvJ`hgm)XvDqIr}6ZgW;wXzt6149EqtQ+rNV7PB{ZY*7go)yh89#qo*#p}GL%#fb`;iXuoO>n^R(X<cXOuCLLJ!$~1us)k@yVV-S;)dikM?Xo;|UAv@vnn(n3s0Cz7yV`&So2)fKYo{*4ZNYVu@J3FXEBFmwgkNuK2vR8&(vWg+YMCwK{CnFR%Z(?M&Y>{gv<-%1#h@(dH!S0-d0!dt6AxN`CZSC1LK1spklKmCKfiLPOO^r*_p<B6k|R>GV(L2*_>hc?$=GzoK)@R6^I@<1R_KjSD~XV6M)dEGQowB1^mPQ1yc`kF6C9({Go02fRGHM!=x><N4M(3%{*NO4A$TNdUDB|Eo>SR-u+$pipV2#Z-|kKJszTagF&@vBgA8F_lCkZlM3ldPVy?JZWA*+jR2ItTLvjD!Oc<iPuaV*qc{f&2;@=DC`(oO+>63tv;56@kgHz_%0SCsR62y!A`pxH(wVDciB&kkXQOqVlvKMapd1Fg-uLCC#>#o@`Vkf&hi>byot$0zMdTV1s(-?In3UX9zQ$7x?DY{@X$)PJwFN*b=hN}piIpxf%<p`NjYJApAsb#+Q^7JiKj<nEVaPD#>jW%&8a}7#FS4%gCg2cmIX1kR=-@x$NFwWR+kdxh#hAEO;!31pAP@`a=i$@)&nrofe%{%gvtP<rSJbO2F6?y4iO}TOks->8OM+2-Sw9%BZWjnRjl>VJ;zEx7(UaW3(k|^|sxcQ@yu3^Z1?Th`9ZY)l|e3hhlrBkxg>|!TU1!hU4v*G>vGY})k8!#d%(*x4rn~uCua04zxDsxU1-86l^>KGZ9qx8v6zXikuOu?}>zYV=figT8g8#_{^*Om4IthK9OTyzpt-|7irDsZze8OS&#ooY_(k8BWLp%|Rt^psk|&yZMv^p}$z1J5B^8=&(7#a0&#4dH}wMb!6<t&=B)rUed75I)HrMZ{WGl9E?@;Iy)8{#o<-u(}IZGF&yQH^};UGAV(gc~#H`SNs|w*?b`7->ZOC))7tFEk5_zzUNKGNBgGf`ILr2+G@8dXCwqwLgpCrKiao9^-lHA+*U}hQK|(1VY%GWyZ8qZ9t2Moj5j8Jbbdoem7c7;U?V*lLu*_%l&`xu6Bl77k8z7%Yr<F}*upD#B9VY<k2197beJP7ey^k~tm#wA;W~SZy{*T2y-w>6i&gOt0olM?U36?nVSt@i0lsd{LV+hzt*oGsOpZ}S*LHb#lu8;B=9`7#kVp<N^}^YjuJm<5ze>Bpw3!8T+`(lW{Qy#T&>r84Qi)ExzVSL-)BvT?D1>Yd`Oc#7)#K9pDj6VBt+9f6USKuOFwtcuN>sp!#9SwdCjwJ+3X`cpb)^hlx0G{d8Uz8OPpUD)IvUIqq62&4;B5$S4PipYshKFcNm(@5UkqUF9Mq!~3*h}p$D2CWz(DPQ1v?y1T8FqV_cT0^o<ya1-2ZzVbeXz|wIn|~*hEnPj`JCRU{wP`Dx8vsqIrx7esW4b;~X%#W#!3^l0=T2WjqUT3)jYVg~Ib)(X2_)gzhLHV`CYngOX>bG1mk#PzD#8pGS3nrZ%6pUG?8;<7OxrP(;m)kuoSvoV-e9Y(DWTq90Y>SGTK7R|17#RJJzO%8Z+}A)dD6#91*Ma?#gWHoL(xm~~Jb@=92~(QN-w=$KmKf{^jX921Tv;!`bg$Ib;9{mbZ;Qp@3C*vPcWn^&d9`MB*r9-3bjE`l=NEEA28NB%pbw6Vngq8<*do-s$)_52|V3Sej?&=Q>L^02oRso-_!VGTAD&9QcoQITvmbPjlC6=tq$gN5J-p*R6YHaf{OHWzh;a!s~cIa9Rsc?66DS+Kp?+jdwt1_n}q{zPtKmjEdqp{uK$%9y%1Hvs@0aRHIZU<BW5oi6h6bCf)SSRhr3!E@v6TsHO5`a%C9d_6%79JWJUd-FC=e3EjB^I^5h&=~EDU`RJ}q~YnTE1*Jb)9}(oU#ENVjFRW$(Hak^lw45<*VJR7z#F*{oF+XYD>*c&T|%#;gaBg?+Dy(DmOnwIUL-LB?=^xCPooJ>0WW*uFpq9jxs(Nkn6K@~m7Wgn|BcFshowY(gySi}ulY7rOf2#~I4q09zVHKqXPW4MCoMu7hl0;9T%YFH{KPdoH$Wu2yzSc_df~pM^;62Pbuy#%nWjQ2v{9R6faG)l={wz+%JL#GrA~+Vpiu_1H2NB_B|>{qszxivfIrjXYH`*qM}7!Pqm(G;uIm~ZV_vu1inKF_HZ*c5)#oT{=NlM@)hg}n*!pl6I*;aBN^s}RI7{All3oCIQV}KXb{A|mg`q47;BLm%XP)+{t!pb|)W0XL#P)OP28QhiJlXNQPEI$|USCK<=C%z6X{_8STvLw%81gm+!iyvY$coYmo0CvH!9#&wV;YX?6I|uoE0+|GJue{)+1uQVtU**XX~82XBkZ<bCCzy7PIg;>u%hc|4-$)E10{QxwVV;Olh(F~Bo1WgJG08h?4;7%5bW$<4EIb8P+C1(>e8RTBZDQAi78WB9K<Ecuqc^^bs!f@`PEUraWEKFurs1^LT+-pID`@dhr3$U4vp_QuOfo$InjX(Z+<n4A-Wwa`WaUcaI6?1X&Ek__n}K9BimBXAOo$T7^G+{&WHp|mZp-4PYg>8h|?>;T99qP_4{jNsW>ha^wETWb1QT@3VOEDNuK+#lPFK?F&*p@8(m&ci}8G&s>ICA>>xfg<6ex$T`0GqM`6a%srHOTSzlq<6v=L57G`4EPcQWBTa-X3v!NX&6#h;MPsc<;>G{U<_<L>A1=S*|>w;lMHhTa`wyk)aLXp@47eb@@#wv1Dw!MI)O6VfRN8uqDJEfd*1RxRxQ%4bCaY3V-3a~=F6%Q1M-b;j5I(>W73DhM}c$OtWENgXdOt>Tl5|n}}%2+RCgD<UhSn9}x3b`Vvazg3d4&W?lsowFMo%-%SvPO&$Lb=iMA-pfojRtQNQ|4sSck2N;9!V(i`{OCf8m$lw-VSm@;d0<HW;x;LUZg=xdw{Oah~3M<#SWT7ujxv5Vm*Zdttb&gUC@MBf*~HSs@Cqu_@c~fX9}!BF%Fn{-lCzG225If)N~p0cu^UDi#IDk99~<UMZzYLAgyEC53rQUxhoh~-5{{LFP%dMz`K;hhhi*yTCsrmzAlQO@tWgq)ZYNQV9_!f+y;y6TOo0u;7rgXs8Z3oC6p}dp6x|J_)~YYCF@k$v<ihJ4ilpQ=LJlaX0)7(MdDc``?7!uLx8izHx!{E%TUFY-=GvB*jiC>gBpHYPrjYvCdme6dX5#vcFMpWEbpV&8!Ok&#a@!x01lmQ0_aG;^n`m-VVV>)=}D>BG>~mYtfsJ8z~8ui{+dY_ui27?v0$*6RuVP1P8OdD$O=bXn!TQ&Sf1?DSCJ&Q`vdWA%!W-nuM5!57FH=CH=A{IW1Nyg+Is0QAanu%Ff>NdwKx~*MFZz;kL{s9{V@k=z_l%V4$z2&cDFKIl&7yu)%+1EAWlvunNVo?Y_{hMKnHw;Px(N0p<^yyN;lRm2TY?`!bf@}4~VybjN#Jqp-lrj##5}kj*ccvv~yZ$pV(;zC3JWwagONOUbJ#$yfmbsS`li4_#zOD7vz;wa#=0bcS*Y>@nm5qye%=xF3f1+W#-|wdcYH{44Mf~o4b_+Zn^D0F2p<y7)Ic<=IGrB{)MMcF-|maSpm06C5fOwFr65+nZXCG>xD7092W3$qveHY0$@}-6vVRNgUah}Ia*XEnw}gD{axCo%6kE^i=-bbboi-I7>zuf3-l!jFw@qO<^E_8aJjLF9S&y)1Go8&tihKXeYdgDp3>|{jZws!#47q;T&wR;2~S#do|2P%Roy`hjX_}0KCRy)v(h0D!O}fwo&GPGdGQTR=QspXzpK75^=CUuy<oQmaLK8_J7=+$@E;C_kX{bpjDSlt9pmZvf*{n&Y!C8elu+)dU>*gQ20NbCYqyZE=_{c%gmUy$a{)*kVjErQm0#Is_84j;ylyGU#`B2Q&Y${lG(1h3N|6YVB&bps593@9$~F;EnBEWjdin~)hhi8y%^p5FBbPEh(W6L(u|2Mm;D4xhH*fpCPg(K^L4>Uk8^v>s+93q;{jyf3eAVjh_OdwNRH3ERH;R!)(r{9tPc7w9Rp_VKLpBg)Q&~nON1Q?`NT(p*@Vw(mP?h1a_?#u$sDShkeNTYiI$CKB5mE-sb!r3|K~Wd*I%}{stHJ4&a<H~SfhkuYJ{cO}>4tA1t+At6B;^Q(o{&-JdYZ0rCU1S^2<L7#0+L~nF{>qJwt9MQR2Wo-YLVFqOYK8l_8s5w`Mq?`cu+RS1|$idXDrqEUZ4&14fd$Qi1E1<e;^@-GmAj!8LTUFpp-UVl)V@LoDJbSN7Bd9voUl}j67;ep(f2sIpXAwc!hk8NXPhX1-hES>W!`ptWgE}&!?#`5mW@JjAq2aZ!jYYmu^7Pu#3xtd*JDB>Me`OW?>i%yI|6!VaBLlvls1o_7Y71Ta8LM-NxEf5qc}sf6~)Hk|7{eDtS5Ag_}gMa-!uQv||Rg$y(G))((bj5>Yr&+9L*&PV5EIzR}+c7VZEJi1I>!u#L8eaqC<T9{q>z;6kjn)f>a82!tW1svW094X+O1<PG&!aj%08%^JV04OHDE(DbH!t<Y}l@|gBS;(!)O1>u?$HymsZ+LKTsqtb_|a<O%jTm{7ugZL8~sxc{E!Gjj!$iORb;hZbbpR9X202n8RJ}$L`b|+Zj3XP-~mg{o}GZXEHtwNh|3b`>kG_z#L;XOmJk{Y@3rH0A?F^NG)R<CsVLkUzWDr0l9)5y?*pP>rkzL|vdO`(N&^?yS!{B2Ptp(h;UQh}PW>@_K!Gih2hhbnX~xT`Z`t%w>HDH6VJk3*K7^U5aTI3HiOu9GUX8jUmBjFMB1nSr%jx*6*D!GfR>>42>Ds?9eBQ|P^aD<2Q8Pf>^lQc}Kv&p6{g)2#N|X9*!u7AGJNHNqcplc76okaIAf?^C?RA_Y=~Qug9;H+Gu&vL~<;O@%@{3S3|Em9qX=A7F;)I#PY4rOs$`EUE;Pt3q2n*v2`8fDx%8|F}a7#0Z9=fSZu){h5?hrt;at<dSHaDude!?Rrhg{mk3j$kqce7RpgxSHs;^Jt3=DlT)V5K~{{*PKlgPw(g!<;WPlAtiNGyO#+URYA@W0rAl3@`n||fG-cd5Ob%kIG)Ge>jW-bmQt&DqNw(FE>RH}7{K<H33mVppC5xO!+S5I4cy81<7lthG${KH&pstx-_c!1+c}?WY)4cgSN_l$q?W*WZ{iH$}oke&`1*)(wiczc0SS{VbP`3bq1kXa6UFIeH5HlY<;{e_d(632fcM2;+<ieGrvxr8dLe9vls^lW@HFQ4v%SRvXP2EXfc!@W8Cu}>raeW2X2_9%@W6N7N)4Bv(JHjn{af?NsFZibw)Jakhw$cvFO3l&^OFI~%#pm+GiOtY%X!~q6bXf9~(Qa{Ko>h0!f^n!e5n!@#b(a~gcwJ{tm_)D3vu!c~suA6Lg7QrdD)r?C8%P}i9pQIaIOA^?A7&@JpYhuSX+p-gm3rz>FdFTy_v(J^4ZJtWm+UoF5LrW<+={_qZ8f6>*GgQS<H#4wR<+cpyb#WFy6m<jpcnviwNB<>SxoI;sEuOktHs-<4ku)L2~srGuvC00tDfU&ab6Yg3(&>Du9X6gNQC8qs1I;ht(_Ci47tP7vj8_3q`^RCj?>GsB)ma4066yuv#8TTW4LE6?KMuIr9-CiJ<t-^1LQnlsZc8iw*EJiTw*Q8I35%{W-ixFSm~s=j(E2(Ic~J=h?3_m*KxX)C3Dl5U&kMjeEPD2uvCX-=4rI2<ijS-XV+Smt=Ogvjj4o=AG~UFaDkWnK2lQYBWR*(q1_fJcW@~1q?>+-5;&@rYoIX-`Qsg~2JJ;U1yls1@^F;5ai|Q!W|3_~toybS0C*KnENIL_!m3Q$Onw2%{_AqS0|^>^sb?cdKZ^?=^*Ltgc(R8Z&?d9XQ$OZwr4SLq+4Sj8N~z910Sr2Tsm9ASa;Row;exeJ2YhqnN14uqB@OdfCX8lX=<#47Z>~h}w&$|txrUmKUH+mg-+t`)V&G&($11U@a@EX2I4g=z+lr0P{^hQJ+<09EX+bHi#6@g}o;($eJ}B*_<QMq_65o~;B^_x<=Qs-MKvt>Bj|3Q!`XZXv$R<2RqLE6mo}`fVe}#bPoIr|?DsZ3!H(2%<JA)9)&W2FrJK|=IvA(xfD8=f_=zG=Fp&2)Y1Rd}z>H;^Lt2MfzmN+yBYl-2ANq#Ti9<r1A8GF{R)X6^N|4HIbs3_rxf}tN;7cP(DSPhp&T{sRLafMk3Tjy?<3Y{u?m$mGyiinB|LYu;kq58m89GYZ@74#l9<;L-(o`JB?*)bfi%Vt+Vm|dmbbunbL#adpF`co%)%t``$A$O}@wRa~h;-Tx$rTz!%JckR3|0rKxayac0tmS3}8WnGyDm&{C@12$xN>G`mKm;#=sZEJ5kqCerO7j*Kg{y)ti%kn)N<gFaJuZA~pfcuBwYa(~UMf=yFJ80UNk=b){JQ=To6?_@v?skrD%bH%*|GN1PFb=7X1xlcXolB{&dEdVn4wkExL+%FL5fZ5qg;SW9J{hlq!6DL-S|~iiN1EL){5ezzc2q~H6n}&!mtRN6J>sjuX^G>Mg+@T<EVPO9O*`pw^;uda@m}~PcLH2$<513CMd6O1(AtU9%~88wMj9eUIl>%C<f1^qnmC=)G*PxV7g<<!q6pc#Av-roQ6fk9-jB?ID|^L?>@bK_vYu<_rFQt@p8Cen=w>H%k6b@54~I4Wc)b2vD!nqa)O|utu;qXLMN<(&J>?szabcdse%R%)q0@K+iPSo(jtxqK8F9uPoWBf5H)}yexBXLEwodcxoCFd>cT1u1JA9Mq{scHu5PEN<--M3;vJYC7E~>L&dyO%{|q)!jMU6VK=K`oS77`{H~CWFeNZMuk6l{enuHB^ydk-8J+HzyRio4E7@RLzG{#*8%Ym$fLCgW8k~$3k;qZe|`nC$c#@E}U+GY}{-Be;6RUqXRA=`V)(`q4@!VLceAv?f$1&f=rQO#Pn3J%4Ef|V4F`hP9_(x#0p3K%!PCxe7&nEAiyNbkQ`dDyh9W^c%5xV1DlX=!GSBVwKr>?LXyL4h{IQ?cx<GsjMUWevV?r}0L=W^K&Sq~2P`)}&sqs%BJu4ZSlor|GKF{8TiwQCt~av(T!Vr#;OS2@^%aX{Nf09dUD$oliuvxiC7Ow^Vb)@D(Oc)Nrjlys)6iARw){2Guq88WC~r40ssOoMT$Gk&5R;6KGWRv+}=aC6Xw5g?sSv1B%-+#Sj44<@YrvuT?>tIB}4Tkk$vgN?1{H=wXR{OxDP5a4Jw%;*aTc<4sK1yeyP#%%Je+Hv_){?PSdRj3W12kZjuGRsq}*ldTSw{j&UpjG+B;9X2W1p`Wi^v_%P3#T8{n`}m=uCFC79d&_q0a`d!n)5}b7%#k7}f>~how%l|)r?dFsd1kJSh7V43oihah%6&RHEY7};Lw*%D->$lfBYnO5AB?suXc?y;{rrE%HUg9')).decode("utf-8"))
_FR_ITEMS = ()
_FR_STATE = {
    0: {"last_step": -1, "due_step": -1, "due": {}},
    1: {"last_step": -1, "due_step": -1, "due": {}},
}
_WEED_STATE = {0: {}, 1: {}}
_WEED_REPLAY_STEPS = 8
_ANIMAL_PLACE_STATE = {0: {}, 1: {}}
_SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}


def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {
        "farmer": list(action.get("farmer") or ["PASS"]),
        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],
        "market": [list(order) for order in (action.get("market") or [])],
    }


def _seat(obs):
    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0


def _farm(obs, seat):
    farms = list(_get(obs, "farms", []) or [])
    return farms[seat] if seat < len(farms) else {}


def _align_hands(action, obs):
    action = _copy_action(action)
    expected = len(_get(_farm(obs, _seat(obs)), "hands", []) or [])
    hands = list(action.get("hands") or [])
    if len(hands) < expected:
        hands.extend([["PASS"] for _ in range(expected - len(hands))])
    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]
    return action


def _tile_at(farm, position):
    try:
        x, y = int(position[0]), int(position[1])
        return (_get(farm, "tiles", []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return "LOCKED"


def _trace_actor_action(step, actor):
    trace = _ACTIONS[min(max(int(step), 0), len(_ACTIONS) - 1)] or {}
    if actor == "farmer":
        return list(trace.get("farmer") or ["PASS"])
    hands = trace.get("hands", []) or []
    return list(hands[actor] if actor < len(hands) else ["PASS"])


def _weed_repair_action(obs, action, step):
    action = _align_hands(action, obs)
    seat = _seat(obs)
    game = _WEED_STATE[seat]
    if step == 0 or step < int(game.get("last_step", -1)):
        game = {"last_step": step, "active": {}}
        _WEED_STATE[seat] = game
    game["last_step"] = step
    farm = _farm(obs, seat)
    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]
    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    active = game.setdefault("active", {})

    for actor, transaction in list(active.items()):
        index = 0 if actor == "farmer" else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        age = step - int(transaction["start"])
        if age == 1:
            unit_actions[index] = list(transaction["intended"])
        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:
            unit_actions[index] = _trace_actor_action(step - 1, actor)
        else:
            active.pop(actor, None)

    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = "farmer" if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or not intended:
            continue
        if intended[0] not in ("BUILD_PASTURE", "PLANT"):
            continue
        tile = _tile_at(farm, position)
        if not isinstance(tile, dict) or tile.get("kind") != "WEED":
            continue
        active[actor] = {"start": step, "intended": list(intended)}
        unit_actions[index] = ["DIG"]

    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]
    action["hands"] = unit_actions[1:]
    return _align_hands(action, obs)


def _animal_place_repair_action(obs, action, step):
    """Close a carried-animal transaction when the trace targets bare soil.

    The repair is deliberately bounded to a free following trace slot, so the
    actor builds the required structure and places its already-owned animal
    without shifting any subsequent route action.
    """
    action = _align_hands(action, obs)
    seat = _seat(obs)
    game = _ANIMAL_PLACE_STATE[seat]
    if step == 0 or step < int(game.get("last_step", -1)):
        game = {"last_step": step, "active": {}}
        _ANIMAL_PLACE_STATE[seat] = game
    game["last_step"] = step
    farm = _farm(obs, seat)
    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]
    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    inventories = list(_get(_get(obs, "private", {}) or {}, "inventories", []) or [])
    active = game.setdefault("active", {})
    required_kind = {"COW": "PASTURE", "SHEEP": "PASTURE", "GOOSE": "COOP"}

    for actor, transaction in list(active.items()):
        index = 0 if actor == "farmer" else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        animal = transaction["animal"]
        inventory = inventories[index] if index < len(inventories) else {}
        tile = _tile_at(farm, positions[index]) if index < len(positions) else "LOCKED"
        if (
            isinstance(tile, dict)
            and tile.get("kind") == required_kind[animal]
            and not tile.get("animal")
            and int(_get(inventory, animal, 0) or 0) > 0
        ):
            unit_actions[index] = ["PLACE", animal, 1]
        active.pop(actor, None)

    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = "farmer" if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or len(intended) < 2:
            continue
        animal = str(intended[1]).upper()
        if intended[0] != "PLACE" or animal not in required_kind:
            continue
        inventory = inventories[index] if index < len(inventories) else {}
        if int(_get(inventory, animal, 0) or 0) <= 0 or _tile_at(farm, position) is not None:
            continue
        if _trace_actor_action(step + 1, actor)[0] != "PASS":
            continue
        active[actor] = {"animal": animal}
        unit_actions[index] = ["BUILD_" + required_kind[animal]]

    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]
    action["hands"] = unit_actions[1:]
    return _align_hands(action, obs)


def _fr_state(obs, step):
    seat = _seat(obs)
    state = _FR_STATE[seat]
    if step == 0 or step < int(state.get("last_step", -1)):
        state = {"last_step": step, "due_step": -1, "due": {}}
        _FR_STATE[seat] = state
    state["last_step"] = step
    if 0 <= int(state.get("due_step", -1)) < step:
        state["due_step"], state["due"] = -1, {}
    return state


def _town_demand_now(obs, item, step):
    demand = 1 if item != "FERTILIZER" and step % 24 == 0 else 0
    if step % 4 != 0:
        return demand
    town = _get(obs, "town", {}) or {}
    for shop in list(_get(town, "unlocked_shops", []) or []):
        products = _SHOP_PRODUCTS.get(shop, ())
        if item in products:
            demand += 2 if len(products) == 1 else 1
    return demand


def _future_quantity(step, item):
    future = step + 1
    if not 0 <= future < len(_ACTIONS):
        return 0
    return sum(
        max(0, int(order[2]))
        for order in (_ACTIONS[future].get("market") or [])
        if len(order) >= 3 and order[0] == "SELL" and order[1] == item
    )


def _pickup_reserve(action, item):
    reserve = 0
    for order in [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]:
        if isinstance(order, (list, tuple)) and len(order) >= 2 and order[0] == "PICKUP" and order[1] == item:
            try:
                reserve += max(0, int(order[2])) if len(order) >= 3 else 1
            except (TypeError, ValueError):
                reserve += 1
    return reserve


def _existing_sell(action, item):
    return sum(
        max(0, int(order[2]))
        for order in (action.get("market") or [])
        if len(order) >= 3 and order[0] == "SELL" and order[1] == item
    )


def _repay(action, state, step):
    if int(state.get("due_step", -1)) != step:
        return action
    due = {str(item): max(0, int(quantity)) for item, quantity in dict(state.get("due", {})).items()}
    action = _copy_action(action)
    market = []
    for raw in action.get("market") or []:
        order = list(raw)
        if len(order) >= 3 and order[0] == "SELL" and order[1] in due and due[order[1]] > 0:
            requested = max(0, int(order[2]))
            reduction = min(requested, due[order[1]])
            requested -= reduction
            due[order[1]] -= reduction
            if requested <= 0:
                continue
            order[2] = requested
        market.append(order)
    action["market"] = market[:10]
    state["due_step"], state["due"] = -1, {}
    return action


def _front_run(action, obs, state, step):
    if not _FR_ITEMS:
        return action
    private = _get(obs, "private", {}) or {}
    shed = _get(private, "shed", {}) or {}
    moved = {}
    action = _copy_action(action)
    for item in _FR_ITEMS:
        target = _future_quantity(step, item)
        if target <= 0 or _town_demand_now(obs, item, step) > 0:
            continue
        stock = max(0, int(_get(shed, item, 0) or 0))
        reserve = _pickup_reserve(action, item) + _existing_sell(action, item)
        quantity = min(target, max(0, stock - reserve))
        if quantity <= 0:
            continue
        market = [list(order) for order in (action.get("market") or [])]
        existing = next((order for order in market if len(order) >= 3 and order[0] == "SELL" and order[1] == item), None)
        if existing is not None:
            existing[2] = max(0, int(existing[2])) + quantity
        elif len(market) < 10:
            market.append(["SELL", item, quantity])
        else:
            continue
        action["market"] = market[:10]
        moved[item] = moved.get(item, 0) + quantity
    if moved:
        state["due_step"] = step + 1
        state["due"] = moved
    return action



_AM_ITEMS = ("STRAWBERRY", "MILK", "WOOL")
_AM_BASE_PRICE = {"STRAWBERRY": 120.0, "MILK": 160.0, "WOOL": 200.0}
_AM_CONFIG = {'start_step': 456, 'reserve_early': 12, 'reserve_mid': 8, 'reserve_late': 6, 'reserve_tail': 0, 'shed_soft': 72, 'shed_hard': 88, 'shed_reserve_cut': 7, 'price_gate': 0.66, 'pressure_trigger': 2.0, 'pressure_gate_cut': 0.18, 'capacity_trigger': 0, 'capacity_gate_cut': 0.08, 'tranche': 4, 'pressure_tranche': 4, 'shed_tranche': 4, 'tail_tranche': 8, 'demand_reserve': 2, 'demand_reserve_cap': 14, 'demand_gate': 0.045, 'demand_gate_cap': 0.24, 'max_extra_per_item': 24, 'mirror_latch_step': 289, 'mirror_composition_distance': 2, 'mirror_money_distance': 250, 'mirror_max_extra_per_item': 0}
_AM_STATE = {
    0: {"last_step": -1, "last_inventory": {}, "last_sold": {}, "last_shops": (), "pressure": {}, "added": {}, "near_mirror": None},
    1: {"last_step": -1, "last_inventory": {}, "last_sold": {}, "last_shops": (), "pressure": {}, "added": {}, "near_mirror": None},
}


def _am_town_demand(shops, item, step):
    demand = 1 if item != "FERTILIZER" and step % 24 == 0 else 0
    if step % 4 != 0:
        return demand
    for shop in shops:
        products = _SHOP_PRODUCTS.get(shop, ())
        if item in products:
            demand += 2 if len(products) == 1 else 1
    return demand


def _am_shop_demand_units(shops, item):
    demand = 0
    for shop in shops:
        products = _SHOP_PRODUCTS.get(shop, ())
        if item in products:
            demand += 2 if len(products) == 1 else 1
    return demand


def _am_count(farm, kind):
    total = 0
    for row in (_get(farm, "tiles", []) or []):
        for tile in row or []:
            if not isinstance(tile, dict):
                continue
            if kind == "STRAWBERRY" and tile.get("kind") == "PLANT" and tile.get("crop") == kind:
                total += 1
            elif kind in {"COW", "SHEEP"} and tile.get("animal") == kind:
                total += 1
    return total


def _am_public_capacity(obs, item):
    farms = list(_get(obs, "farms", []) or [])
    seat = _seat(obs)
    if len(farms) < 2:
        return 0
    own = farms[seat]
    opponent = farms[1 - seat]
    kind = {"STRAWBERRY": "STRAWBERRY", "MILK": "COW", "WOOL": "SHEEP"}[item]
    return _am_count(opponent, kind) - _am_count(own, kind)


def _am_macro_signature(farm):
    return {
        "COW": _am_count(farm, "COW"),
        "SHEEP": _am_count(farm, "SHEEP"),
        "STRAWBERRY": _am_count(farm, "STRAWBERRY"),
        "hands": len(_get(farm, "hands", []) or []),
        "quadrants": len(_get(farm, "unlocked_quadrants", []) or []),
        "money": float(_get(farm, "money", 0) or 0),
    }


def _am_near_mirror(obs):
    farms = list(_get(obs, "farms", []) or [])
    seat = _seat(obs)
    if len(farms) < 2:
        return False
    own = _am_macro_signature(farms[seat])
    opponent = _am_macro_signature(farms[1 - seat])
    composition_distance = sum(
        abs(int(own[item]) - int(opponent[item]))
        for item in ("COW", "SHEEP", "STRAWBERRY")
    )
    return (
        composition_distance <= int(_AM_CONFIG["mirror_composition_distance"])
        and own["hands"] == opponent["hands"]
        and own["quadrants"] == opponent["quadrants"]
        and abs(own["money"] - opponent["money"]) <= float(_AM_CONFIG["mirror_money_distance"])
    )


def _am_reset(step):
    return {
        "last_step": step,
        "last_inventory": {},
        "last_sold": {},
        "last_shops": (),
        "pressure": {},
        "added": {},
        "near_mirror": None,
    }


def _am_observe(obs, step):
    seat = _seat(obs)
    state = _AM_STATE[seat]
    if step == 0 or step <= int(state.get("last_step", -1)):
        state = _am_reset(step)
        _AM_STATE[seat] = state

    market = _get(obs, "market", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    if int(state.get("last_step", -1)) == step - 1:
        previous_inventory = dict(state.get("last_inventory", {}))
        previous_sold = dict(state.get("last_sold", {}))
        previous_shops = tuple(state.get("last_shops", ()))
        for item in _AM_ITEMS:
            delta = int(_get(inventory, item, 0) or 0) - int(previous_inventory.get(item, 0) or 0)
            external = (
                delta
                + _am_town_demand(previous_shops, item, step - 1)
                - int(previous_sold.get(item, 0) or 0)
            )
            old = float(dict(state.get("pressure", {})).get(item, 0.0) or 0.0)
            # A decaying public-market signal.  Negative inferred flow does not
            # become a buy fingerprint; it simply lets old pressure decay.
            state.setdefault("pressure", {})[item] = max(0.0, old * 0.72 + min(24.0, max(0.0, float(external))))
    state["last_step"] = step
    if state.get("near_mirror") is None and step >= int(_AM_CONFIG["mirror_latch_step"]):
        state["near_mirror"] = bool(_am_near_mirror(obs))
    return state


def _am_reserve(step, item, total_shed):
    if step < 528:
        reserve = _AM_CONFIG["reserve_early"]
    elif step < 600:
        reserve = _AM_CONFIG["reserve_mid"]
    elif step < 648:
        reserve = _AM_CONFIG["reserve_late"]
    elif step < 684:
        reserve = _AM_CONFIG["reserve_tail"]
    elif step < 704:
        reserve = 2
    else:
        reserve = 0
    if total_shed >= _AM_CONFIG["shed_hard"]:
        reserve = 0
    elif total_shed >= _AM_CONFIG["shed_soft"]:
        reserve = max(0, reserve - _AM_CONFIG["shed_reserve_cut"])
    return reserve


def _am_add_sell(action, item, quantity):
    if quantity <= 0:
        return action
    action = _copy_action(action)
    market = [list(order) for order in (action.get("market") or [])]
    existing = next(
        (
            order
            for order in market
            if len(order) >= 3 and order[0] == "SELL" and order[1] == item
        ),
        None,
    )
    if existing is not None:
        existing[2] = max(0, int(existing[2])) + quantity
    elif len(market) < 10:
        # Premium cash is made available before hires and purchases, while the
        # relative order among existing scheduled orders is untouched.
        market.insert(0, ["SELL", item, quantity])
    action["market"] = market[:10]
    return action


def _adaptive_market(action, obs, step):
    state = _am_observe(obs, step)
    private = _get(obs, "private", {}) or {}
    shed = _get(private, "shed", {}) or {}
    market = _get(obs, "market", {}) or {}
    prices = _get(market, "prices", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    town = _get(obs, "town", {}) or {}
    shops = tuple(_get(town, "unlocked_shops", []) or [])
    total_shed = sum(max(0, int(value or 0)) for value in dict(shed).values())

    if step >= _AM_CONFIG["start_step"]:
        for item in _AM_ITEMS:
            stock = max(0, int(_get(shed, item, 0) or 0))
            pickup = _pickup_reserve(action, item)
            scheduled = min(max(0, stock - pickup), _existing_sell(action, item))
            unscheduled = max(0, stock - pickup - scheduled)
            reserve = _am_reserve(step, item, total_shed)
            shop_demand = _am_shop_demand_units(shops, item)
            if step < 684:
                reserve += min(
                    int(_AM_CONFIG["demand_reserve_cap"]),
                    shop_demand * int(_AM_CONFIG["demand_reserve"]),
                )
            excess = max(0, unscheduled - reserve)
            if excess <= 0:
                continue

            already_added = int(dict(state.get("added", {})).get(item, 0) or 0)
            maximum_extra = (
                int(_AM_CONFIG["mirror_max_extra_per_item"])
                if bool(state.get("near_mirror", False))
                else int(_AM_CONFIG["max_extra_per_item"])
            )
            budget = max(0, maximum_extra - already_added)
            if budget <= 0:
                continue

            ratio = float(_get(prices, item, 0) or 0) / _AM_BASE_PRICE[item]
            pressure = float(dict(state.get("pressure", {})).get(item, 0.0) or 0.0)
            capacity_delta = _am_public_capacity(obs, item)
            gate = float(_AM_CONFIG["price_gate"])
            if step < 684:
                gate += min(
                    float(_AM_CONFIG["demand_gate_cap"]),
                    shop_demand * float(_AM_CONFIG["demand_gate"]),
                )
            if step >= 648:
                gate -= 0.12
            if step >= 684:
                gate -= 0.18
            if pressure >= _AM_CONFIG["pressure_trigger"]:
                gate -= _AM_CONFIG["pressure_gate_cut"]
            if capacity_delta >= _AM_CONFIG["capacity_trigger"]:
                gate -= _AM_CONFIG["capacity_gate_cut"]

            urgent = total_shed >= _AM_CONFIG["shed_hard"] or step >= 704
            if not urgent and ratio < max(0.20, gate):
                continue

            tranche = int(_AM_CONFIG["tranche"])
            if pressure >= _AM_CONFIG["pressure_trigger"]:
                tranche += int(_AM_CONFIG["pressure_tranche"])
            if total_shed >= _AM_CONFIG["shed_soft"]:
                tranche += int(_AM_CONFIG["shed_tranche"])
            if step >= 684:
                tranche += int(_AM_CONFIG["tail_tranche"])
            quantity = min(excess, max(1, tranche), budget)
            action = _am_add_sell(action, item, quantity)
            state.setdefault("added", {})[item] = int(state.setdefault("added", {}).get(item, 0) or 0) + quantity

    # Save the observable transition ingredients only after the final action is
    # known, so next turn's external-flow estimate does not peek at the rival.
    actual = {}
    for item in _AM_ITEMS:
        stock = max(0, int(_get(shed, item, 0) or 0))
        actual[item] = min(stock, _existing_sell(action, item))
    state["last_inventory"] = {item: int(_get(inventory, item, 0) or 0) for item in _AM_ITEMS}
    state["last_sold"] = actual
    state["last_shops"] = shops
    return action

def agent(obs):
    try:
        fallback = int(_get(obs, "day", 0) or 0) * 24 + int(_get(obs, "hour", 0) or 0)
        step = min(max(0, int(_get(obs, "step", fallback) if _get(obs, "step", None) is not None else fallback)), len(_ACTIONS) - 1)
        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), step)
        action = _animal_place_repair_action(obs, action, step)
        state = _fr_state(obs, step)
        action = _repay(action, state, step)
        action = _front_run(action, obs, state, step)
        action = _adaptive_market(action, obs, step)
        return _align_hands(action, obs)
    except Exception:
        farm = _farm(obs, _seat(obs))
        return {
            "farmer": ["PASS"],
            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],
            "market": [],
        }
'''


In [3]:
from pathlib import Path
import hashlib
import json
OUTPUT = Path.cwd()
EXPECTED_SOURCE_SHA = 'de6a93ea927114826ae0e6b03cd359ba49ed29498a442958d1840ddabbf52427'
NORMALIZED_SHA = '2c8e9269ff1208b44e6bdac45c16324ab49f066a3bf6031c7be7023abecf0039'
CRLF_LINES = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558]
source_bytes = SOURCE.replace("\r\n", "\n").encode("utf-8")
if hashlib.sha256(source_bytes).hexdigest() == NORMALIZED_SHA:
    lines = source_bytes.splitlines(keepends=True)
    source_bytes = b"".join(line[:-1] + b"\r\n" if i in CRLF_LINES and line.endswith(b"\n") else line
                            for i, line in enumerate(lines))
SOURCE_SHA = hashlib.sha256(source_bytes).hexdigest()
IS_HISTORICAL = SOURCE_SHA == EXPECTED_SOURCE_SHA
compile(source_bytes, "main.py", "exec")
(OUTPUT / "main.py").write_bytes(source_bytes)
def fresh_agent():
    namespace = {"__name__": "baseline"}
    exec(compile(source_bytes, "main.py", "exec"), namespace)
    return namespace
print("Source:", "exact historical S59" if IS_HISTORICAL else "MODIFIED — no historical score attribution")
print("SHA-256:", SOURCE_SHA)


## 4. Test the transaction boundary

These small tests isolate the repair by replacing only its schedule with two
synthetic slots. The default file written above is never changed. Passing a
guard test means that branch behaves as described; it does not establish game
strength. The final-hour example deliberately documents an unresolved failure.


In [4]:
import copy

guard_checks = []
def check(label, condition):
    assert condition, label
    guard_checks.append(label)

def fixture(animal="COW", stock=1, tile=None, next_action=None, seat=0, hand=False):
    policy = fresh_agent()
    idle = {"farmer":["PASS"], "hands":[["PASS"]], "market":[]}
    policy["_ACTIONS"] = [copy.deepcopy(idle) for _ in range(720)]
    index = 1 if hand else 0
    intended = copy.deepcopy(idle)
    if hand:
        intended["hands"][0] = ["PLACE", animal, 1]
        policy["_ACTIONS"][71]["hands"][0] = next_action or ["PASS"]
    else:
        intended["farmer"] = ["PLACE", animal, 1]
        policy["_ACTIONS"][71]["farmer"] = next_action or ["PASS"]
    farm = {"farmer":[0,0], "hands":[[0,0]] if hand else [], "tiles":[[tile]]}
    obs = {"player":seat,"farms":[copy.deepcopy(farm),copy.deepcopy(farm)],
           "private":{"inventories":[{},{}] if hand else [{}]}}
    obs["private"]["inventories"][index] = {animal:stock}
    return policy, obs, intended, index

def actor_action(action,index):
    return action["farmer"] if index==0 else action["hands"][index-1]

for animal, structure in (("COW","PASTURE"),("SHEEP","PASTURE"),("GOOSE","COOP")):
    for seat in (0,1):
        for hand in (False,True):
            p, obs, intended, idx = fixture(animal=animal,seat=seat,hand=hand)
            original = copy.deepcopy(intended)
            first = p["_animal_place_repair_action"](obs,intended,70)
            prefix = f"{animal}/seat{seat}/{'hand' if hand else 'farmer'}"
            check(prefix+": build",actor_action(first,idx)==["BUILD_"+structure])
            check(prefix+": input unchanged",intended==original)
            obs["farms"][seat]["tiles"][0][0] = {"kind":structure,"animal":None}
            second = p["_animal_place_repair_action"](obs,p["_ACTIONS"][71],71)
            check(prefix+": place",actor_action(second,idx)==["PLACE",animal,1])
            third = p["_animal_place_repair_action"](obs,p["_ACTIONS"][72],72)
            check(prefix+": route resumes",actor_action(third,idx)==["PASS"])

for label, options in (
    ("not carried",{"stock":0}),
    ("occupied tile",{"tile":{"kind":"PASTURE","animal":"COW"}}),
    ("weed tile",{"tile":{"kind":"WEED"}}),
    ("busy next slot",{"next_action":["MOVE","N"]}),
    ("unsupported animal",{"animal":"PIG"}),
):
    p, obs, intended, idx = fixture(**options)
    result = p["_animal_place_repair_action"](obs,intended,70)
    check("refuse: "+label,actor_action(result,idx)==intended["farmer"])

for label,tile,stock in (
    ("build failed",None,1),
    ("animal no longer carried",{"kind":"PASTURE"},0),
    ("structure occupied",{"kind":"PASTURE","animal":"SHEEP"},1),
):
    p,obs,intended,idx=fixture()
    p["_animal_place_repair_action"](obs,intended,70)
    obs["farms"][0]["tiles"][0][0]=tile
    obs["private"]["inventories"][0]["COW"]=stock
    result=p["_animal_place_repair_action"](obs,p["_ACTIONS"][71],71)
    check("completion guard: "+label,result["farmer"]==["PASS"])

# A synthetic final-hour transition: the engine may return cargo at day end.
# This checks the missing guard in S59, not a repaired future variant.
p,obs,intended,idx=fixture()
first=p["_animal_place_repair_action"](obs,intended,95)
check("known limitation: build at final hour",first["farmer"]==["BUILD_PASTURE"])
obs["private"]["inventories"][0]["COW"]=0
obs["farms"][0]["tiles"][0][0]={"kind":"PASTURE"}
second=p["_animal_place_repair_action"](obs,p["_ACTIONS"][96],96)
check("known limitation: no cow placed after boundary",second["farmer"]==["PASS"])
print("Placement:",len(guard_checks),"checks passed, including the documented boundary limitation.")

def market_fixture(stock, pickup=0, scheduled=0, full=False):
    p=fresh_agent()
    # One farm avoids activating the inherited near-mirror guard in this isolated fixture.
    obs={"player":0,"farms":[{"tiles":[],"hands":[]}],
         "private":{"shed":{"MILK":stock}},"market":{"prices":{"MILK":160},"inventory":{}},
         "town":{"unlocked_shops":[]}}
    action={"farmer":["PICKUP","MILK",pickup] if pickup else ["PASS"],"hands":[],
            "market":[["SELL","MILK",scheduled]] if scheduled else []}
    if full:
        action["market"]=[["BUY","WHEAT",1] for _ in range(10)]
    return p,obs,action

for stock in (0,1,5,20,40):
    for pickup in (0,3):
        for scheduled in (0,5):
            p,obs,action=market_fixture(stock,pickup,scheduled)
            before=p["_existing_sell"](action,"MILK")
            result=p["_adaptive_market"](action,obs,704)
            extra=p["_existing_sell"](result,"MILK")-before
            available=max(0,stock-pickup-min(max(0,stock-pickup),scheduled))
            check(f"extra-stock bound {stock}/{pickup}/{scheduled}",0<=extra<=available)
            check(f"queue bound {stock}/{pickup}/{scheduled}",len(result["market"])<=10)
p,obs,action=market_fixture(40)
result=p["_adaptive_market"](action,obs,704)
check("positive control: extras actually added",p["_existing_sell"](result,"MILK")>0)
for step in range(705,712):
    p["_adaptive_market"](action,obs,step)
check("cumulative extra budget",p["_AM_STATE"][0]["added"]["MILK"]<=p["_AM_CONFIG"]["max_extra_per_item"])
p,obs,action=market_fixture(40,full=True)
result=p["_adaptive_market"](action,obs,704)
check("full queue stays unchanged",result==action)
check("known limitation: full queue consumes budget",p["_AM_STATE"][0]["added"]["MILK"]>0)
print("Total:",len(guard_checks),"guard checks passed.")
print("Known limitations reproduced: day boundary; full-queue budget accounting.")


## 5. Complete-game checks

The next cell plays two fixed seeds in both player positions against the official
starter. It checks 720 states, DONE/DONE, 719 callbacks and agreement between the
named `agent` function and the environment's file loader on every call. A second,
exception-transparent copy of the same entrypoint detects errors that the
historical fallback would otherwise hide.

The starter is a compatibility opponent. Banks below are local game rewards,
not leaderboard ratings. For strategy changes, use fresh reacting opponents,
both seats and a held-out panel; do not tune against these four games.


In [5]:
SMOKE_PROGRAM = '"""Fresh-process compatibility checks; never sends anything to Kaggle."""\nimport ast\nimport contextlib\nimport copy\nimport io\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nseeds=json.loads(sys.stdin.read())\nwith contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):\n    from kaggle_environments import make\n    from kaggle_environments.agent import get_last_callable\nraw=Path(\'main.py\').read_bytes()\ntext=raw.decode(\'utf-8\')\ntree=ast.parse(text)\nentry=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name==\'agent\')\nassert isinstance(entry.body[0],ast.Try)\n# Expose the same try-body in a separate module; do not change the packaged file.\nentry.body=entry.body[0].body\nast.fix_missing_locations(tree)\nstrict_code=compile(tree,\'strict_test_copy\',\'exec\')\nrows=[]\nfor seed in seeds:\n    for seat in (0,1):\n        module={\'__name__\':\'s59_test\'}\n        strict_module={\'__name__\':\'s59_strict_test\'}\n        exec(compile(raw,\'main.py\',\'exec\'),module)\n        exec(strict_code,strict_module)\n        loader=get_last_callable(text)\n        assert loader.__name__==\'agent\'\n        calls=[0]\n        timings=[]\n        def checked(obs,config):\n            start=time.perf_counter()\n            result=module[\'agent\'](copy.deepcopy(obs))\n            assert result==loader(copy.deepcopy(obs))==strict_module[\'agent\'](copy.deepcopy(obs))\n            timings.append((time.perf_counter()-start)*1000)\n            assert set(result)=={\'farmer\',\'hands\',\'market\'}\n            assert len(result[\'market\'])<=10\n            assert len(result[\'hands\'])==len(obs[\'farms\'][seat][\'hands\'])\n            calls[0]+=1\n            return result\n        env=make(\'kaggriculture\',configuration={\'episodeSteps\':720,\'seed\':seed},debug=False)\n        env.run([checked,\'starter\'] if seat==0 else [\'starter\',checked])\n        assert [s.status for s in env.state]==[\'DONE\',\'DONE\']\n        assert len(env.steps)==720 and calls[0]==719\n        rows.append({\'seed\':seed,\'seat\':seat,\'bank\':env.state[seat].reward,\n                     \'opponent_bank\':env.state[1-seat].reward,\'callbacks\':calls[0],\n                     \'max_three_way_check_ms\':round(max(timings),3)})\nprint(\'S59_RESULT:\'+json.dumps(rows))\n'
result = subprocess.run([sys.executable, "-c", SMOKE_PROGRAM], input=json.dumps(SMOKE_SEEDS),
                        capture_output=True, text=True, timeout=300)
if result.returncode:
    print((result.stdout + result.stderr)[-3000:])
    raise RuntimeError("Full-game check failed; do not submit this build")
records = [line[len("S59_RESULT:"):] for line in result.stdout.splitlines() if line.startswith("S59_RESULT:")]
assert len(records) == 1
game_results = json.loads(records[0])
for row in game_results:
    print("seed={seed} seat={seat} | DONE | bank={bank} vs {opponent_bank} | {callbacks} matched calls".format(**row))


## 6. Build and verify the submission

The agent archive contains `main.py`, `LICENSE.txt` and `NOTICE.txt`, so the
license and attribution travel with the downloaded agent. They are also available
as separate outputs. `build_receipt.json` records source identity and the checks from
this run. Archive packaging metadata can differ from the original upload;
the historical identity check concerns the exact `main.py` bytes.


In [6]:
import gzip
import io
import tarfile
LICENSE_TEXT = '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n'
NOTICE_TEXT = 'S59 Stock & Placement notebook, 2026, Dmitrii Gluzdov. Apache-2.0.\nDerived from Boatlee V29-R1 Adaptive Market Hysteresis, version 1, script 346682136.\nhttps://www.kaggle.com/code/boatlee/v29-r1-adaptive-market-hysteresis?scriptVersionId=346682136\nBoatlee attributes its production trace to RngRng submission 55948382 and public episodes 104679155, 104689164 and 104691666.\nS59 adds carried-animal placement repair and changes reserve_tail, tail_tranche and max_extra_per_item.\nThis edition adds executable synthetic guard/counterexample tests and source/loader validation.\nThe default main.py is unchanged historical S59; its existing text is preserved.\nAI assistance was used in implementation, testing and notebook preparation.\nNo affiliation with or endorsement by the upstream authors is claimed.\n'
(OUTPUT / "LICENSE.txt").write_text(LICENSE_TEXT, encoding="utf-8")
(OUTPUT / "NOTICE.txt").write_text(NOTICE_TEXT, encoding="utf-8")
buffer = io.BytesIO()
package_files = {"main.py":source_bytes,"LICENSE.txt":LICENSE_TEXT.encode("utf-8"),
                 "NOTICE.txt":NOTICE_TEXT.encode("utf-8")}
with tarfile.open(fileobj=buffer, mode="w", format=tarfile.USTAR_FORMAT) as archive:
    for name, data in package_files.items():
        member = tarfile.TarInfo(name)
        member.size = len(data)
        member.mode = 0o644
        member.mtime = 0
        archive.addfile(member, io.BytesIO(data))
packed = io.BytesIO()
with gzip.GzipFile(fileobj=packed, filename="", mode="wb", mtime=0) as compressed:
    compressed.write(buffer.getvalue())
archive_bytes = packed.getvalue()
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
    assert archive.getnames() == list(package_files)
    for name, data in package_files.items():
        assert archive.extractfile(name).read() == data
(OUTPUT / "submission.tar.gz").write_bytes(archive_bytes)
receipt = {"historical_source":IS_HISTORICAL,"source_sha256":SOURCE_SHA,
           "archive_sha256":hashlib.sha256(archive_bytes).hexdigest(),"engine":ENGINE_PIN,
           "guard_checks":guard_checks,"games":game_results,"submitted":False}
(OUTPUT / "build_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print("Ready:", "submission.tar.gz", "|", len(archive_bytes), "bytes")
print("No leaderboard submission was made.")


## What the historical score does — and does not — tell us

The original qualification recorded 40 wins from 40 local games, followed by
2 serial checks and a 4-game holdout without losses. The upload-validation replay
matched all 719 actions in both seats. These are dated development results,
not results freshly reproduced by this notebook.

The later 146-game ladder audit was less tidy: **77 wins, 68 losses and one tie**.
It also found 68 cows left in terminal sheds and 168 lost milk units. A sibling
with more conservative tail settings had a much better win rate but met a very
different opponent schedule. The rating difference does not isolate the value
of our placement repair or of the three market settings.

The useful takeaway is narrower: check whether a requested transaction can
finish, test the refusal conditions, and keep extra sales inside observed stock.
The next experiment would add a physical-turn/day-boundary check, then compare
matched full games with the unchanged baseline. That modified agent would need
its own evaluation; it would not inherit S59's historical score.

### Attribution and scope

Boatlee's Apache-2.0 code supplies the complete production schedule and market
controller. Its published RngRng attribution is preserved in `NOTICE.txt`.
No additional replay files, opponent observations or private workspace data are
distributed. The examples here are synthetic; the historical counts above come
from our dated S59 development records. AI assistance was used for implementation,
testing and preparation of this notebook.
